<a href="https://colab.research.google.com/github/Simiformes/zadania-rekrutacyjne/blob/main/Zadanie2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip uninstall -y torchao
!pip install -U ms-swift datasets rouge_score sacrebleu "qwen_vl_utils>=0.0.14"

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 72.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 28.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 46.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 152.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import json
from datasets import load_dataset

# 1. Pobranie zbioru danych z HuggingFace
raw_dataset = load_dataset("medalpaca/medical_meadow_wikidoc_patient_information", split="train")

filtered_data = []

# 2. Filtracja zgodnie z kryteriami z zadania
for item in raw_dataset:
    inst = item.get('instruction', '') or ''
    inp = item.get('input', '') or ''
    out = item.get('output', '') or ''

    full_input = f"{inst}\n{inp}".strip() if inp else inst.strip()

    # Puste przykłady
    if len(full_input) == 0 or len(out) == 0:
        continue
    # Output poniżej 20 znaków
    if len(out) < 20:
        continue
    # Łączna długość przekraczająca 1500 znaków
    if len(full_input) + len(out) > 1500:
        continue

    filtered_data.append({"query": full_input, "response": out})

print(f"Liczba przykładów przed filtracją: {len(raw_dataset)}")
print(f"Liczba przykładów po filtracji: {len(filtered_data)}")

# 3. Wydzielenie zbioru testowego (100 przykładów) i treningowego
test_data = filtered_data[:100]
train_data = filtered_data[100:]

# Zapis zbioru treningowego do formatu JSONL akceptowanego przez MS Swift
with open("train_data.jsonl", "w", encoding="utf-8") as f:
    for entry in train_data:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

README.md:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

medical_meadow_wikidoc_patient_info.json: reconstructing file:   0%|          |  0.00B / 3.49MB            

medical_meadow_wikidoc_patient_info.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5942 [00:00<?, ? examples/s]

Liczba przykładów przed filtracją: 5942
Liczba przykładów po filtracji: 5701


In [7]:
import os
import numpy as np
import sacrebleu
from rouge_score import rouge_scorer
from swift.infer_engine import TransformersEngine, RequestConfig, InferRequest

# Inicjalizacja modelu bazowego
model_id = 'qwen/Qwen3.5-2B'
engine_base = TransformersEngine(model_id)

def evaluate_model(engine, test_samples, max_eval=50):
    scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

    predictions = []
    references = []
    word_counts = []

    samples_to_eval = test_samples[:max_eval]

    for item in samples_to_eval:
        prompt = item['query']
        ref = item['response']

        request_config = RequestConfig(max_tokens=256, temperature=0.0)
        infer_requests = [InferRequest(messages=[{'role': 'user', 'content': prompt}])]

        resp_list = engine.infer(infer_requests, request_config)
        pred = resp_list[0].choices[0].message.content.strip()

        predictions.append(pred)
        references.append(ref)
        word_counts.append(len(pred.split()))

    # Obliczanie ROUGE
    r1, r2, rl = [], [], []
    for p, r in zip(predictions, references):
        scores = scorer.score(r, p)
        r1.append(scores['rouge1'].fmeasure)
        r2.append(scores['rouge2'].fmeasure)
        rl.append(scores['rougeL'].fmeasure)

    # Obliczanie BLEU
    bleu = sacrebleu.corpus_bleu(predictions, [[r for r in references]]).score

    results = {
        'ROUGE-1': round(float(np.mean(r1)), 4),
        'ROUGE-2': round(float(np.mean(r2)), 4),
        'ROUGE-L': round(float(np.mean(rl)), 4),
        'BLEU': round(float(bleu), 4),
        'Średnia liczba słów': round(float(np.mean(word_counts)), 2)
    }

    return results, predictions, references

# Ewaluacja przed treningiem
results_before, preds_before, refs = evaluate_model(engine_base, test_data, max_eval=50)
print("Wyniki przed tuningiem:", results_before)

[INFO:swift] Downloading the model from ModelScope Hub, model_id: qwen/Qwen3.5-2B
2026-10-28 11:39:14,176 | INFO    | modelscope_hub.download | Downloading 14 files from qwen/Qwen3.5-2B@master


Downloading:   0%|          | 0/14 [00:00<?, ?file/s]

.gitattributes:   0%|          | 0.00/2.21k [00:00<?, ?B/s]

LICENSE:   0%|          | 0.00/11.3k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.75k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/modelscope/models/qwen--Qwen3.5-2B/snapshots/master
[WARNING:swift] Loading model from '/root/.cache/modelscope/models/qwen--Qwen3.5-2B/snapshots/master' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
[INFO:swift] Setting torch_dtype: torch.bfloat16
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[WARNING:swift] Loading config from '/root/.cache/modelscope/models/qwen--Qwen3.5-2B/snapshots/master' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[WARNING:swift] Loading tokenizer/processor from '/root/.cache/modelscope/models/qwen--Qwen3.5-2B/snapshots/master' with trust_remote_code=True. Custom Py

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[INFO:swift] Create the template for the infer_engine
[INFO:swift] default_system: None
[INFO:swift] max_length: 262144
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000
[INFO:swift] Setting ROOT_IMAGE_DIR: None. You can adjust this hyperparameter through the environment variable: `ROOT_IMAGE_DIR`.
[INFO:swift] Setting QWENVL_BBOX_FORMAT: legacy. You can adjust this hyperparameter through the environment variable: `QWENVL_BBOX_FORMAT`.
[INFO:swift] Successfully registered `/usr/local/lib/python3.13/dist-packages/swift/dataset/data/dataset_info.json`.
[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much s

Wyniki przed tuningiem: {'ROUGE-1': 0.2461, 'ROUGE-2': 0.0502, 'ROUGE-L': 0.1304, 'BLEU': 1.6532, 'Średnia liczba słów': 168.86}


In [11]:
!swift sft \
    --model qwen/Qwen3.5-2B \
    --tuner_type lora \
    --dataset train_data.jsonl \
    --torch_dtype bfloat16 \
    --num_train_epochs 1 \
    --per_device_train_batch_size 2 \
    --gradient_accumulation_steps 4 \
    --learning_rate 1e-4 \
    --lora_rank 8 \
    --lora_alpha 32 \
    --max_length 1024 \
    --output_dir output \
    --use_hf true

run sh: `/usr/bin/python3 /usr/local/lib/python3.13/dist-packages/swift/cli/sft.py --model qwen/Qwen3.5-2B --tuner_type lora --dataset train_data.jsonl --torch_dtype bfloat16 --num_train_epochs 1 --per_device_train_batch_size 2 --gradient_accumulation_steps 4 --learning_rate 1e-4 --lora_rank 8 --lora_alpha 32 --max_length 1024 --output_dir output --use_hf true`
[INFO:swift] Successfully registered `/usr/local/lib/python3.13/dist-packages/swift/dataset/data/dataset_info.json`.
[INFO:swift] rank: -1, local_rank: -1, world_size: 1, local_world_size: 1
[INFO:swift] Downloading the model from HuggingFace Hub, model_id: qwen/Qwen3.5-2B
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files: 100% 12/12 [00:00<00:00, 1607.01it/s]
Download complete:           |  0.00B            
Reconstruction complete: |          |  0.00B /  0.00B                         [INFO:swift] Loading the model using model_dir: /root/.cache/huggingface/hub/models--qwen--Qwen3.

In [16]:
import glob

# Automatyczne odnalezienie ścieżki do zapisanego checkpointu LoRA
checkpoint_dirs = glob.glob("output/v2-20261028-123105/checkpoint-701")
latest_checkpoint = sorted(checkpoint_dirs)[-1]
print(f"Wczytywanie LoRA z: {latest_checkpoint}")

# Ładowanie silnika z zaaplikowanym adapterem LoRA
engine_lora = TransformersEngine(model_id, adapters=[latest_checkpoint])

# Ewaluacja po tuningu
results_after, preds_after, _ = evaluate_model(engine_lora, test_data, max_eval=50)
print("Wyniki po tuningu:", results_after)

[INFO:swift] Downloading the model from ModelScope Hub, model_id: qwen/Qwen3.5-2B


Wczytywanie LoRA z: output/v2-20261028-123105/checkpoint-701


2026-10-28 15:47:57,423 | INFO    | modelscope_hub.download | Downloading 14 files from qwen/Qwen3.5-2B@master


Downloading:   0%|          | 0/14 [00:00<?, ?file/s]

[INFO:swift] Loading the model using model_dir: /root/.cache/modelscope/models/qwen--Qwen3.5-2B/snapshots/master
[WARNING:swift] Loading model from '/root/.cache/modelscope/models/qwen--Qwen3.5-2B/snapshots/master' with trust_remote_code=True. This will execute custom Python code shipped in the model repository (e.g. configuration_*.py, modeling_*.py, tokenization_*.py). Only proceed if you trust the source of this model.
[INFO:swift] Setting torch_dtype: torch.bfloat16
[WARNING:swift] Please install the package: `pip install "decord" -U`.
[WARNING:swift] Loading config from '/root/.cache/modelscope/models/qwen--Qwen3.5-2B/snapshots/master' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use models from sources you trust.
[WARNING:swift] Loading tokenizer/processor from '/root/.cache/modelscope/models/qwen--Qwen3.5-2B/snapshots/master' with trust_remote_code=True. Custom Python code from the model repository will be executed. Only use mo

Loading weights:   0%|          | 0/617 [00:00<?, ?it/s]

[INFO:swift] Create the template for the infer_engine
[INFO:swift] Successfully loaded output/v2-20261028-123105/checkpoint-701/args.json.
[INFO:swift] default_system: None
[INFO:swift] max_length: 262144
[INFO:swift] response_prefix: None
[INFO:swift] agent_template: qwen3_5
[INFO:swift] norm_bbox: norm1000


Wyniki po tuningu: {'ROUGE-1': 0.2581, 'ROUGE-2': 0.1031, 'ROUGE-L': 0.1904, 'BLEU': 5.1888, 'Średnia liczba słów': 71.82}


In [17]:
import pandas as pd

# Podsumowanie wyników w tabeli
df_results = pd.DataFrame([results_before, results_after], index=["Przed tuningiem", "Po tuningu"])
print("=== ZESTAWIENIE WYNIKÓW EWALUACJI ===")
display(df_results)

# Wypisanie 2-3 przykładowych odpowiedzi
print("\n=== PRZYKŁADOWE ODPOWIEDZI MODELU ===")
for i in range(3):
    print(f"\n--- PRZYKŁAD {i+1} ---")
    print(f"PYTANIE: {test_data[i]['query']}")
    print(f"ODPOWIEDŹ WZORCOWA (REFERENCE): {refs[i]}")
    print(f"ODPOWIEDŹ PRZED TUNINGIEM: {preds_before[i]}")
    print(f"ODPOWIEDŹ PO TUNINGU: {preds_after[i]}")

=== ZESTAWIENIE WYNIKÓW EWALUACJI ===


,ROUGE-1,ROUGE-2,ROUGE-L,BLEU,Średnia liczba słów
Przed tuningiem,0.2461,0.0502,0.1304,1.6532,168.86
Po tuningu,0.2581,0.1031,0.1904,5.1888,71.82



=== PRZYKŁADOWE ODPOWIEDZI MODELU ===

--- PRZYKŁAD 1 ---
PYTANIE: Answer this question truthfully
What are the symptoms of Allergy?
ODPOWIEDŹ WZORCOWA (REFERENCE): Allergy symptoms vary, but may include:
Breathing problems (coughing, shortness of breath) Burning, tearing, or itchy eyes Conjunctivitis (red, swollen eyes) Coughing Diarrhea Headache Hives Itching of the nose, mouth, throat, skin, or any other area Runny nose Skin rashes Stomach cramps Vomiting Wheezing
What part of the body is contacted by the allergen plays a role in the symptoms you develop. For example:
Allergens that are breathed in often cause a stuffy nose, itchy nose and throat, mucus production, cough, or wheezing. Allergens that touch the eyes may cause itchy, watery, red, swollen eyes. Eating something you are allergic to can cause nausea, vomiting, abdominal pain, cramping, diarrhea, or a severe, life-threatening reaction. Allergens that touch the skin can cause a skin rash, hives, itching, blisters, or even 